Q1. Prepare a facility bookings reporting dataset as the following.
```
member_id | first_name | last_name | facility_id | slots | start_time
---------------------------------------------------------------------------
```
The report must meet the following criteria.
1. Facility bookings made by a person whose last name is Smith
2. He has booked more than 5 slots in a single booking
3. Report should be sorted by first name of the member in ascending order and number of slots in descending order


1.1 Try answering with SQL

* Join Expression
* Join type
* Column name ambiguity

In [0]:
%sql
select m.member_id, m.first_name, m.last_name, b.facility_id, b.slots, b.start_time
from dev.spark_db.bookings b
inner join dev.spark_db.members m 
on m.member_id = b.member_id
where m.last_name = "Smith" and b.slots > 5
order by m.first_name asc, b.slots desc

In [0]:
from pyspark.sql.functions import expr, col

members_df = spark.table("dev.spark_db.members").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")

#join_expr = expr("m.member_id == b.member_id")
join_expr = col("m.member_id") == col("b.member_id")
reports_df = (
    members_df.join(bookings_df, join_expr, "inner")
    .filter("m.last_name == 'Smith' and b.slots > 5" )
    .select(
        "m.member_id",
        "m.first_name",
        "m.last_name",
        "b.facility_id",
        "b.slots",
        "b.start_time")
    .orderBy("m.first_name", col("b.slots").desc())
)

reports_df.display()




In [0]:
%sql
select * From dev.spark_db.facilities

In [0]:
from pyspark.sql.functions import col, expr

members_df = spark.table("dev.spark_db.members").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")
facilities_df = spark.table("dev.spark_db.facilities").alias("f")

join_expr = col("m.member_id") == col("b.member_id")
join_expr_2 = col("f.facility_id") == col("b.facility_id")

reports_df =(
    members_df.join(bookings_df, join_expr, "inner")
    .join(facilities_df, join_expr_2, "inner")
    .where("m.last_name == 'Smith' and b.slots > 5" )
    .select(
        "m.member_id",
        "m.first_name",
        "m.last_name",
        "b.facility_id",
        "f.facility_name",
        "b.slots",
        "b.start_time",
        (col("f.member_cost") * col("b.slots")).alias("booking_amount"))
    .orderBy(col("m.first_name").asc(),
            col("booking_amount").desc())
)

reports_df.display()
